# 21 — Bootstrap Confidence Interval / 自助置信区间

**Chapter 21 — File 4 of 4**

## Summary / 摘要

Bootstrap confidence intervals provide non-parametric estimates without assuming a specific population distribution. By resampling the original data with replacement and computing the statistic of interest (e.g., mean) for each resample, the empirical distribution of the statistic emerges. The confidence interval bounds are extracted as percentiles (e.g., 2.5th and 97.5th for 95% CI). This approach is flexible, works for any statistic, and doesn't require mathematical derivations of sampling distributions.

自助置信区间提供了无需假设特定总体分布的非参数估计。通过有放回地重新采样原始数据并为每个重采样计算感兴趣的统计量（例如均值），统计量的经验分布出现。置信区间边界被提取为百分位数（例如95% CI的2.5th和97.5th）。此方法灵活，适用于任何统计量，不需要对采样分布进行数学推导。

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Generate Sample Data / 生成样本数据

In [ ]:
# Generate original sample data
# 生成原始样本数据
original_data = np.array([23, 26, 30, 34, 36, 37, 38, 39, 40, 41,
                         42, 43, 44, 45, 45, 46, 47, 48, 49, 50])

# Calculate sample statistics
# 计算样本统计量
sample_mean = np.mean(original_data)
sample_std = np.std(original_data, ddof=1)
sample_size = len(original_data)

# Display data information
# 显示数据信息
print(f"Original sample data: {original_data}")
print(f"\nSample statistics:")
print(f"  Sample size: {sample_size}")
print(f"  Mean: {sample_mean:.4f}")
print(f"  Std Dev: {sample_std:.4f}")
print(f"  Min: {np.min(original_data)}")
print(f"  Max: {np.max(original_data)}")

## Step 3 — Perform Bootstrap Resampling / 执行自助重采样

In [ ]:
# Bootstrap parameters
# 自助参数
n_bootstrap = 100  # Number of bootstrap samples / 自助样本数
confidence_level = 0.95  # 95% confidence / 95%置信
alpha = 1 - confidence_level

# Store bootstrap means
# 存储自助均值
bootstrap_means = []

# Perform bootstrap resampling
# 执行自助重采样
for i in range(n_bootstrap):
    # Resample with replacement
    # 有放回地重采样
    resample = np.random.choice(original_data, size=sample_size, replace=True)
    
    # Calculate mean of resample
    # 计算重采样的均值
    bootstrap_means.append(np.mean(resample))

# Convert to array
# 转换为数组
bootstrap_means = np.array(bootstrap_means)

# Display bootstrap statistics
# 显示自助统计量
print(f"\nBootstrap Statistics ({n_bootstrap} samples):")
print(f"  Mean of bootstrap means: {np.mean(bootstrap_means):.4f}")
print(f"  Std of bootstrap means: {np.std(bootstrap_means):.4f}")
print(f"  Min bootstrap mean: {np.min(bootstrap_means):.4f}")
print(f"  Max bootstrap mean: {np.max(bootstrap_means):.4f}")

## Step 4 — Calculate Bootstrap Confidence Interval / 计算自助置信区间

In [ ]:
# Calculate percentiles for confidence interval
# 计算置信区间的百分位数
lower_percentile = (alpha / 2) * 100  # e.g., 2.5 for 95% CI
upper_percentile = (1 - alpha / 2) * 100  # e.g., 97.5 for 95% CI

# Extract CI bounds using percentiles
# 使用百分位数提取CI边界
ci_lower_bootstrap = np.percentile(bootstrap_means, lower_percentile)
ci_upper_bootstrap = np.percentile(bootstrap_means, upper_percentile)

# Display bootstrap CI
# 显示自助置信区间
print(f"\nBootstrap {confidence_level*100:.0f}% Confidence Interval for Mean:")
print(f"  Lower bound ({lower_percentile:.1f}th percentile): {ci_lower_bootstrap:.4f}")
print(f"  Upper bound ({upper_percentile:.1f}th percentile): {ci_upper_bootstrap:.4f}")
print(f"  Interval: [{ci_lower_bootstrap:.4f}, {ci_upper_bootstrap:.4f}]")
print(f"  Width: {ci_upper_bootstrap - ci_lower_bootstrap:.4f}")
print(f"  Sample mean (point estimate): {sample_mean:.4f}")

## Step 5 — Compare Bootstrap CI with Parametric CI / 比较自助置信区间与参数置信区间

In [ ]:
# Calculate parametric CI (assuming normal distribution)
# 计算参数置信区间（假设正态分布）
se = sample_std / np.sqrt(sample_size)
t_critical = stats.t.ppf(1 - alpha/2, df=sample_size-1)  # Use t-distribution for sample
                                                           # 对样本使用t分布
ci_lower_parametric = sample_mean - t_critical * se
ci_upper_parametric = sample_mean + t_critical * se

# Compare the two methods
# 比较两种方法
print(f"\nComparison of CI Methods:")
print(f"{'Method':20} | {'Lower':10} | {'Upper':10} | {'Width':10}")
print("-" * 55)
print(f"{'Bootstrap':20} | {ci_lower_bootstrap:10.4f} | {ci_upper_bootstrap:10.4f} | {ci_upper_bootstrap - ci_lower_bootstrap:10.4f}")
print(f"{'Parametric (t-test)':20} | {ci_lower_parametric:10.4f} | {ci_upper_parametric:10.4f} | {ci_upper_parametric - ci_lower_parametric:10.4f}")

# Calculate differences
# 计算差异
lower_diff = abs(ci_lower_bootstrap - ci_lower_parametric)
upper_diff = abs(ci_upper_bootstrap - ci_upper_parametric)
print(f"\nDifference in bounds:")
print(f"  Lower bound difference: {lower_diff:.4f}")
print(f"  Upper bound difference: {upper_diff:.4f}")

## Step 6 — Visualize Bootstrap Distribution and CI / 可视化自助分布和置信区间

In [ ]:
# Create comprehensive visualization
# 创建综合可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Original data histogram
# 图1: 原始数据直方图
axes[0, 0].hist(original_data, bins=10, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].axvline(sample_mean, color='red', linestyle='--', linewidth=2, label='Sample mean')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Original Sample Data Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Bootstrap means distribution
# 图2: 自助均值分布
axes[0, 1].hist(bootstrap_means, bins=15, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].axvline(sample_mean, color='blue', linestyle='-', linewidth=2, label='Original mean')
axes[0, 1].axvline(ci_lower_bootstrap, color='red', linestyle='--', linewidth=2, label='CI bounds')
axes[0, 1].axvline(ci_upper_bootstrap, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Bootstrap Mean')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Bootstrap Distribution of Means ({n_bootstrap} samples)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Q-Q plot for bootstrap means
# 图3: 自助均值的Q-Q图
stats.probplot(bootstrap_means, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot of Bootstrap Means')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: CI comparison
# 图4: 置信区间比较
methods = ['Bootstrap', 'Parametric']
centers = [sample_mean, sample_mean]
errors = [(ci_upper_bootstrap - ci_lower_bootstrap)/2, (ci_upper_parametric - ci_lower_parametric)/2]

axes[1, 1].errorbar(centers, methods, xerr=errors, fmt='o', markersize=10,
                    capsize=15, capthick=2, linewidth=2, color=['green', 'blue'])
axes[1, 1].set_xlabel('Mean')
axes[1, 1].set_title(f'{confidence_level*100:.0f}% Confidence Interval Comparison')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
## Learning Notes / 学习笔记

- **Statistical Concept**: Bootstrap CI bypasses theoretical distributional assumptions by leveraging the empirical distribution of resampled statistics. The method is justified by the bootstrap principle: the sample relates to the population the same way resamples relate to the original sample. Percentile-based extraction is intuitive and works for any statistic, though bias correction methods (BCa) improve coverage for skewed distributions.
  
  **统计概念**: 自助置信区间通过利用重采样统计的经验分布来绕过理论分布假设。该方法由自助原理证明：样本与总体的关系与重采样与原始样本的关系相同。基于百分位数的提取直观，适用于任何统计量，尽管偏差修正方法(BCa)改进了偏斜分布的覆盖率。

- **ML Application**: Bootstrap CIs are indispensable for complex estimators where theoretical distributions are unknown or hard to derive (e.g., median, quantiles, complex ML metrics). In A/B testing, they handle non-normal metrics directly. For cross-validation and ensemble methods, bootstrap provides uncertainty quantification for performance metrics without assumptions, making it ideal for production ML systems where robustness is critical.
  
  **ML应用**: 自助置信区间对于理论分布未知或难以推导的复杂估计器是不可或缺的（例如中位数、分位数、复杂的ML指标）。在A/B测试中，它们直接处理非正态指标。对于交叉验证和集成方法，自助为性能指标提供了无假设的不确定性量化，使其理想用于鲁棒性至关重要的生产ML系统。

➡️ **Next**: `../chapter_22/01_test_data.ipynb`

## Complete Code / 完整代码一览

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)
original_data = np.array([23, 26, 30, 34, 36, 37, 38, 39, 40, 41,
                         42, 43, 44, 45, 45, 46, 47, 48, 49, 50])

sample_mean = np.mean(original_data)
sample_size = len(original_data)

n_bootstrap = 100
bootstrap_means = []

for i in range(n_bootstrap):
    resample = np.random.choice(original_data, size=sample_size, replace=True)
    bootstrap_means.append(np.mean(resample))

bootstrap_means = np.array(bootstrap_means)

ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

print(f"Sample mean: {sample_mean:.4f}")
print(f"95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")